# Moore-AnimateAnyone -- Colab Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/goldkangsan/moore-animate-anyone/blob/main/colab_animate_anyone.ipynb)

**Reference Image + Driving Video -> Character Animation**

---
### 실행 순서
1. GPU 확인
2. 레포 클론
3. 패키지 설치 + 패치 -> **런타임 재시작**
4. Weights 다운로드
5. 입력 설정
6. Inference 실행
7. 결과 확인


In [ ]:
# Step 1. GPU 확인
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')


In [ ]:
# Step 2. 레포 클론
import os
if not os.path.exists('/content/Moore-AnimateAnyone'):
    !git clone https://github.com/MooreThreads/Moore-AnimateAnyone /content/Moore-AnimateAnyone
else:
    print('Already cloned.')
os.chdir('/content/Moore-AnimateAnyone')
print('Working dir:', os.getcwd())


In [ ]:
# Step 3. 패키지 설치 + 호환성 패치
# !! 실행 후 반드시 런타임 재시작: 메뉴 > 런타임 > 런타임 재시작 !!

# 1) 핵심 패키지 설치
!pip install 'diffusers==0.24.0' 'transformers==4.30.2' --force-reinstall -q
!pip install accelerate einops omegaconf av imageio imageio-ffmpeg decord \
             open-clip-torch scikit-image tqdm -q
!pip install -q 'clip @ https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip'

# 2) hf_cache_home 호환성 패치
#    huggingface_hub 신버전에서 hf_cache_home 이 callable 이거나 없을 수 있음
import glob as _g
PATCH = (
    'try:\n'
    '    from huggingface_hub.constants import hf_cache_home\n'
    '    if callable(hf_cache_home):\n'
    '        hf_cache_home = hf_cache_home()\n'
    'except (ImportError, AttributeError):\n'
    '    import os as _os\n'
    '    hf_cache_home = _os.path.expanduser(\n'
    '        _os.path.join(_os.getenv("HF_HOME",\n'
    '            _os.path.join(_os.getenv("XDG_CACHE_HOME", "~/.cache"), "huggingface")))\n'
    '    )'
)
OLD = 'from huggingface_hub.constants import hf_cache_home'
for _fp in _g.glob('/usr/local/lib/python*/dist-packages/diffusers/utils/constants.py'):
    _txt = open(_fp).read()
    if OLD in _txt and 'callable(hf_cache_home)' not in _txt:
        open(_fp, 'w').write(_txt.replace(OLD, PATCH))
        print(f'[OK] Patched: {_fp}')
    else:
        print(f'[SKIP] Already patched or not found: {_fp}')

# 3) cached_download 호환성 패치
for _fp in _g.glob('/usr/local/lib/python*/dist-packages/diffusers/utils/dynamic_modules_utils.py'):
    _txt = open(_fp).read()
    if 'cached_download' in _txt:
        _txt = _txt.replace(
            'from huggingface_hub import HfFolder, cached_download, hf_hub_download, model_info',
            'from huggingface_hub import HfFolder, hf_hub_download, model_info\ntry:\n    from huggingface_hub import cached_download\nexcept ImportError:\n    from huggingface_hub import hf_hub_download as cached_download'
        )
        open(_fp, 'w').write(_txt)
        print(f'[OK] Patched dynamic_modules_utils: {_fp}')

# 4) pose2vid.py 교체 (configs 모듈 의존성 제거된 버전)
import os
os.makedirs('/content/Moore-AnimateAnyone/scripts', exist_ok=True)
!wget -q -O /content/Moore-AnimateAnyone/scripts/pose2vid.py \
    https://raw.githubusercontent.com/goldkangsan/moore-animate-anyone/main/scripts/pose2vid.py
print('[OK] pose2vid.py 교체 완료')

print()
print('=' * 55)
print('완료! 지금 런타임 재시작 후 Step 4부터 실행하세요')
print('메뉴: 런타임 > 런타임 재시작')
print('=' * 55)


In [ ]:
# Step 4. Weights 다운로드 (런타임 재시작 후 여기서 시작)
import os
os.chdir('/content/Moore-AnimateAnyone')

# diffusers 버전 확인
import diffusers
print(f'diffusers: {diffusers.__version__}')
if not diffusers.__version__.startswith('0.24'):
    print('[WARNING] diffusers 버전이 0.24.x 가 아닙니다. Step 3 재실행 후 런타임 재시작 권장')

!python tools/download_weights.py


In [ ]:
# Step 5. 입력 설정 (원본 레포 샘플 사용)
import urllib.request, os
from PIL import Image
import IPython.display as ipd
os.makedirs('/content/Moore-AnimateAnyone/inputs', exist_ok=True)
REF_NUM  = 10   # 1,2,3,5,10,11 중 선택
POSE_NUM = 2    # 1,2,4,5 중 선택
ref_url  = f'https://raw.githubusercontent.com/MooreThreads/Moore-AnimateAnyone/master/configs/inference/ref_images/anyone-{REF_NUM}.png'
pose_url = f'https://raw.githubusercontent.com/MooreThreads/Moore-AnimateAnyone/master/configs/inference/pose_videos/anyone-video-{POSE_NUM}_kps.mp4'
REF_IMAGE  = f'/content/Moore-AnimateAnyone/inputs/anyone-{REF_NUM}.png'
POSE_VIDEO = f'/content/Moore-AnimateAnyone/inputs/anyone-video-{POSE_NUM}_kps.mp4'
urllib.request.urlretrieve(ref_url,  REF_IMAGE)
urllib.request.urlretrieve(pose_url, POSE_VIDEO)
WIDTH, HEIGHT = 512, 784
def resize_and_crop(image, tw, th):
    ow, oh = image.size
    if ow/oh > tw/th:
        nw = int(oh*tw/th); l = (ow-nw)//2
        image = image.crop((l, 0, l+nw, oh))
    else:
        nh = int(ow*th/tw); t = (oh-nh)//2
        image = image.crop((0, t, ow, t+nh))
    return image.resize((tw, th), Image.LANCZOS)
img = Image.open(REF_IMAGE).convert('RGB')
img_resized = resize_and_crop(img, WIDTH, HEIGHT)
REF_PROCESSED = '/content/Moore-AnimateAnyone/inputs/ref.png'
img_resized.save(REF_PROCESSED)
print(f'REF_PROCESSED : {REF_PROCESSED}  {img_resized.size}')
print(f'POSE_VIDEO    : {POSE_VIDEO}  exists={os.path.exists(POSE_VIDEO)}')
ipd.display(img_resized.resize((200, int(200*HEIGHT/WIDTH))))


In [ ]:
# Step 6. Config 업데이트 + Inference 실행 (T4 기준 5~10분)
import os
from omegaconf import OmegaConf
os.chdir('/content/Moore-AnimateAnyone')

# config 업데이트
cfg = OmegaConf.load('./configs/prompts/animation.yaml')
cfg['test_cases'] = {REF_PROCESSED: [POSE_VIDEO]}
OmegaConf.save(cfg, './configs/prompts/animation.yaml')
print('animation.yaml 업데이트:')
print(OmegaConf.to_yaml(cfg))

# inference 실행
WIDTH=512; HEIGHT=784; N_FRAMES=32; STEPS=30; CFG=3.5; SEED=42
!python scripts/pose2vid.py \
    --config ./configs/prompts/animation.yaml \
    -W {WIDTH} -H {HEIGHT} -L {N_FRAMES} \
    --steps {STEPS} --cfg {CFG} --seed {SEED}


In [ ]:
# Step 7. 결과 확인 + 다운로드
import glob
from IPython.display import Video, display
from google.colab import files
results = sorted(glob.glob('/content/Moore-AnimateAnyone/output/**/*.mp4', recursive=True))
if results:
    latest = results[-1]
    print(f'결과: {latest}')
    display(Video(latest, width=700, embed=True))
    files.download(latest)
else:
    print('결과 없음 - Step 6을 다시 실행하세요')


---
## 트러블슈팅

| 오류 | 해결 |
|------|------|
| `hf_cache_home` TypeError / ImportError | Step 3 재실행 후 런타임 재시작 |
| PositionNet ImportError | Step 3 재실행 후 런타임 재시작 |
| No module named configs | Step 3 재실행 (pose2vid.py 교체 확인) |
| CUDA out of memory | N_FRAMES=16, WIDTH=384, HEIGHT=512 로 낮추기 |
| decord install fail | Step 3에서 무시해도 됨 (imageio 로 fallback) |
